# CMP611 - DNABERT-2 Colab Smoke Test

Bu notebook ile DNABERT-2 fine-tune pipeline'ının Colab GPU'da çalıştığını hızlıca doğrularsın.

## Akış
1. GPU kontrolü
2. Proje klasörünü Drive'dan `/content/project` altına kopyalama
3. Paket kurulumları
4. GUE download + low-data split
5. 1 kısa fine-tune (smoke test)
6. Sonuç JSON kontrolü


In [ ]:
!nvidia-smi || true
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Drive Mount ve Proje Yolunu Ayarla
`PROJECT_SRC_IN_DRIVE` değişkenini kendi Drive yoluna göre düzenle.


In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

# TODO: Gerekirse bu yolu değiştir
PROJECT_SRC_IN_DRIVE = '/content/drive/MyDrive/bio/project'
PROJECT_DIR = '/content/project'

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

shutil.copytree(PROJECT_SRC_IN_DRIVE, PROJECT_DIR)
print('Project copied to:', PROJECT_DIR)
print('Files:', os.listdir(PROJECT_DIR))


In [ ]:
%cd /content/project
!python -m pip install -q -r requirements.txt
!python scripts/check_environment.py


## Smoke Test (1 task, 1 ratio, 1 seed, 1 epoch)
Bu adım uzun sürebilir (dataset indirme + model download).


In [ ]:
%cd /content/project
!bash scripts/colab_smoke_test.sh


In [ ]:
import json, pathlib
result_path = pathlib.Path('/content/project/outputs/smoke_test/results/smoke_prom_core_all_r1_seed13/eval_results.json')
print('Exists:', result_path.exists())
if result_path.exists():
    data = json.loads(result_path.read_text())
    print(json.dumps(data, indent=2))


## (Opsiyonel) Tam Deney Koşusu
Aşağıdaki hücreyi smoke test başarılı olduktan sonra çalıştır:


In [ ]:
%cd /content/project
# !python scripts/run_low_data_experiments.py --config configs/promoter_low_data.yaml
# !python scripts/aggregate_results.py --results-root outputs/experiments --out-csv outputs/summary_all_runs.csv --out-mean-csv outputs/summary_mean_std.csv
